In [1]:
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import wandb

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wandb.init(
    project="Lab1",
    name="transf-roberta",
    config={
        "learning_rate": 2e-5,
        "batch_size": 8,
        "epochs": 3,
        "model": "roberta-base"
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
#charge the dataset
dataset = load_dataset(
    "csv",
    data_files="amazon_cells_labelled_LARGE_25K.txt",
    delimiter="\t",
    column_names=["text", "label"])

# convert labels
dataset = dataset.map(lambda x: {"label": int(x["label"])})

#split
dataset = dataset["train"].train_test_split(test_size=0.3)

# split temp en validation + test
temp_split = dataset["test"].train_test_split(test_size=0.5)

# Tokenizer
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

dataset = {
    "train": dataset["train"],
    "validation": temp_split["train"],
    "test": temp_split["test"]
}

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = {
    split: dataset[split].map(tokenize_function, batched=True)
    for split in dataset
}
data_collator = DataCollatorWithPadding(tokenizer)

Map: 100%|██████████| 3750/3750 [00:00<00:00, 20137.74 examples/s]


In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 689.39it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [6]:
training_args = TrainingArguments(
    output_dir="test-trainer",   
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,          
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=50,
    report_to="wandb"
)



In [8]:
trainer= Trainer(model, training_args, train_dataset= tokenized_datasets["train"], eval_dataset=tokenized_datasets["test"], compute_metrics=compute_metrics,
data_collator= data_collator)
trainer.train()

Step,Training Loss
50,0.158439
100,0.098950
150,0.049979
200,0.260041
250,0.167898
300,0.057699
350,0.217004
400,0.171357
450,0.194384
500,0.115480


Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.29s/it]


TrainOutput(global_step=13125, training_loss=0.08264112669087592, metrics={'train_runtime': 1281.2223, 'train_samples_per_second': 40.976, 'train_steps_per_second': 10.244, 'total_flos': 839790877610640.0, 'train_loss': 0.08264112669087592, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

cm = confusion_matrix(labels, preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.show()

In [9]:
test_results = trainer.evaluate(tokenized_datasets["validation"])
print(test_results)

RuntimeError: on_train_begin must be called before on_evaluate

In [11]:
wandb.finish()

eval/loss,█▁
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
test/loss,▁
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
+3,...
